In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from itables import show
import ipywidgets as widgets
import json
from IPython.display import HTML, display
import altair as alt
import numpy as np
import uuid

import warnings
warnings.filterwarnings("ignore")

data_dir = Path('./data')

df_composition = pd.read_excel(data_dir / 'household_composition_ward.xlsx')
df_size = pd.read_excel(data_dir / 'household_size_ward.xlsx')

# Data Structure

## Household composition

In [2]:
cat_cols = ["main_category", "sub_category1", "sub_category2"]
census_col = "census"

category_presence = (
    df_composition[cat_cols + [census_col]]
    .drop_duplicates()
    .groupby(cat_cols, dropna=False)[census_col]
    .agg(
        presence=lambda x: sorted(x.astype(str).unique()),
        # n_census=lambda x: x.nunique()
    )
    .reset_index()
)

# display(category_presence)

In [3]:
cat_presence_plot = (
    df_composition[["curated_category", "main_category", "census"]]
    .drop_duplicates()
    .copy()
)

cat_presence_plot["census"] = cat_presence_plot["census"].astype(str)

chart = (
    alt.Chart(cat_presence_plot)
    .mark_circle(size=130)
    .encode(
        x=alt.X(
            "census:O",
            title="Census",
            sort=["2001", "2011", "2021"],
            axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y(
            "curated_category:N",
            title=None,
            sort=alt.SortField("main_category", order="ascending")
        ),
        color=alt.Color(
            "main_category:N",
            title="Main category"
        ),
        tooltip=[
            alt.Tooltip("curated_category:N", title="Category"),
            alt.Tooltip("main_category:N", title="Main category"),
            alt.Tooltip("census:O", title="Census")
        ]
    )
    .properties(
        width=300,
        height=540,
        title="Category availability by Census"
    )
)

chart

alt.Chart(...)

## Household size

In [4]:
df_size['main_category'].unique()

array(['Total: All households', '1 person in household',
       '2 people in household', '3 people in household',
       '4 people in household', '5 people in household',
       '6 people in household', '7 people in household',
       '8 or more people in household'], dtype=object)

# Visualisation

## Household composition

In [5]:
trend_table_compo = (
    df_composition[df_composition["sub_category1"].isna()]
    .drop_duplicates()
    .pivot_table(
        index=["ward", "main_category"],
        columns="census",
        values=["number", "pct"],
    )
)

trend_table_compo.columns = [
    f"{value}_{year}" for value, year in trend_table_compo.columns
]

trend_table_compo = trend_table_compo.reset_index()

trend_table_compo["number_change_01_11"] = trend_table_compo["number_2011"] - trend_table_compo["number_2001"]
trend_table_compo["number_change_11_21"] = trend_table_compo["number_2021"] - trend_table_compo["number_2011"]
trend_table_compo["number_change_01_21"] = trend_table_compo["number_2021"] - trend_table_compo["number_2001"]

trend_table_compo["pct_change_01_11"] = trend_table_compo["pct_2011"] - trend_table_compo["pct_2001"]
trend_table_compo["pct_change_11_21"] = trend_table_compo["pct_2021"] - trend_table_compo["pct_2011"]
trend_table_compo["pct_change_01_21"] = trend_table_compo["pct_2021"] - trend_table_compo["pct_2001"]

trend_table_compo = trend_table_compo.sort_values(["ward", "main_category"])

# # show(trend_table)

# ward_dropdown = widgets.Dropdown(
#     options=sorted(trend_table_compo["ward"].dropna().unique()),
#     description="Ward:",
#     layout=widgets.Layout(width="400px")
# )

# def show_ward_trend(ward):
#     df_show = trend_table_compo[trend_table_compo["ward"] == ward].copy()
#     display(df_show)

# widgets.interact(show_ward_trend, ward=ward_dropdown);

In [6]:

trend_json = trend_table_compo.to_json(orient="records")

html = """
<div style="font-family: Arial, sans-serif;">
  <label for="wardSelect"><b>Ward:</b></label>
  <select id="wardSelect" style="width: 320px; margin-bottom: 12px; padding: 4px;"></select>
  <div id="trendTable"></div>
</div>

<script>
(function() {
  const trendData = DATA_PLACEHOLDER;

  const wards = Array.from(new Set(trendData.map(function(d) {
    return d.ward;
  }))).sort();

  const select = document.getElementById("wardSelect");
  const tableDiv = document.getElementById("trendTable");

  wards.forEach(function(w) {
    const option = document.createElement("option");
    option.value = w;
    option.textContent = w;
    select.appendChild(option);
  });

  function isNegativeNumber(value) {
    if (value === null || value === undefined || value === "") {
      return false;
    }

    const cleaned = String(value).replace(/,/g, "");
    const numValue = Number(cleaned);

    return !isNaN(numValue) && numValue < 0;
  }

  function renderTable(ward) {
    const rows = trendData.filter(function(d) {
      return d.ward === ward;
    });

    if (rows.length === 0) {
      tableDiv.innerHTML = "<p>No data available.</p>";
      return;
    }

    const cols = Object.keys(rows[0]);

    let tableHtml = "";
    tableHtml += "<table border='1' style='border-collapse: collapse; width: 100%; font-size: 13px;'>";
    tableHtml += "<thead><tr>";

    cols.forEach(function(c) {
      tableHtml += "<th style='padding: 6px; text-align: left; background-color: #f2f2f2;'>" + c + "</th>";
    });

    tableHtml += "</tr></thead><tbody>";

    rows.forEach(function(row) {
      tableHtml += "<tr>";

      cols.forEach(function(c) {
        let value = row[c];

        if (value === null || value === undefined) {
          value = "";
        }

        let cellStyle = "padding: 6px;";

        if (isNegativeNumber(value)) {
          cellStyle += " color: red; font-weight: 600;";
        }

        tableHtml += "<td style='" + cellStyle + "'>" + value + "</td>";
      });

      tableHtml += "</tr>";
    });

    tableHtml += "</tbody></table>";

    tableDiv.innerHTML = tableHtml;
  }

  select.addEventListener("change", function() {
    renderTable(select.value);
  });

  if (wards.length > 0) {
    select.value = wards[0];
    renderTable(wards[0]);
  }
})();
</script>
"""

html = html.replace("DATA_PLACEHOLDER", trend_json)

display(HTML(html))

In [7]:
df_plot = df_composition[
    df_composition["sub_category1"].isna()
].copy()

ward_dropdown = alt.binding_select(
    options=sorted(df_plot["ward"].dropna().unique()),
    name="Ward: "
)

ward_select = alt.selection_point(
    fields=["ward"],
    bind=ward_dropdown,
    value=df_plot["ward"].dropna().iloc[0]
)

chart = (
    alt.Chart(df_plot)
    .mark_line(point=True)
    .encode(
        x=alt.X("census:O", title="Census"),
        y=alt.Y("pct:Q", title="Percentage"),
        color=alt.Color("main_category:N", title="Category"),
        tooltip=["ward", "main_category", "census", "number", "pct"]
    )
    .add_params(ward_select)
    .transform_filter(ward_select)
    .properties(
        width=650,
        height=400,
        title="Household composition trend by ward"
    )
)

chart

alt.Chart(...)

## Household size

In [8]:
trend_table = (
    df_size
    .pivot_table(
        index=["ward", "main_category"],
        columns="census",
        values=["number", "pct"],
        aggfunc="first"
    )
)

trend_table.columns = [
    f"{value}_{year}" for value, year in trend_table.columns
]

trend_table = trend_table.reset_index()

trend_table["number_change_01_11"] = trend_table["number_2011"] - trend_table["number_2001"]
trend_table["number_change_11_21"] = trend_table["number_2021"] - trend_table["number_2011"]
trend_table["number_change_01_21"] = trend_table["number_2021"] - trend_table["number_2001"]

trend_table["pct_change_01_11"] = trend_table["pct_2011"] - trend_table["pct_2001"]
trend_table["pct_change_11_21"] = trend_table["pct_2021"] - trend_table["pct_2011"]
trend_table["pct_change_01_21"] = trend_table["pct_2021"] - trend_table["pct_2001"]

trend_table = trend_table.sort_values(["ward", "main_category"])

# show(trend_table)

# ward_dropdown = widgets.Dropdown(
#     options=sorted(trend_table["ward"].dropna().unique()),
#     description="Ward:",
#     layout=widgets.Layout(width="400px")
# )

# def show_ward_trend(ward):
#     df_show = trend_table[trend_table["ward"] == ward].copy()
#     display(df_show)

# widgets.interact(show_ward_trend, ward=ward_dropdown);

In [9]:
uid = uuid.uuid4().hex[:8]
select_id = f"wardSelect_{uid}"
table_id = f"trendTable_{uid}"

ward_col = "ward" 

trend_table_clean = trend_table.copy()
trend_table_clean = trend_table_clean.replace([float("inf"), float("-inf")], None)
trend_table_clean = trend_table_clean.where(trend_table_clean.notna(), None)

trend_json = trend_table_clean.to_json(orient="records")

html = f"""
<div style="font-family: Arial, sans-serif;">
  <label for="{select_id}"><b>{ward_col}:</b></label>
  <select id="{select_id}" style="width: 320px; margin-bottom: 12px; padding: 4px;"></select>
  <div id="{table_id}"></div>
</div>

<script>
(function() {{
  const trendData = {trend_json};
  const wardCol = "{ward_col}";

  const wards = Array.from(new Set(
    trendData.map(d => d[wardCol]).filter(x => x !== null && x !== undefined && x !== "")
  )).sort();

  const select = document.getElementById("{select_id}");
  const tableDiv = document.getElementById("{table_id}");

  if (!select || !tableDiv) return;

  if (wards.length === 0) {{
    tableDiv.innerHTML = "<p>No ward values found. Check ward_col.</p>";
    return;
  }}

  wards.forEach(function(w) {{
    const option = document.createElement("option");
    option.value = w;
    option.textContent = w;
    select.appendChild(option);
  }});

  function isNegativeNumber(value) {{
    if (value === null || value === undefined || value === "") return false;
    const numValue = Number(String(value).replace(/,/g, ""));
    return !isNaN(numValue) && numValue < 0;
  }}

  function renderTable() {{
    const rows = trendData.filter(d => d[wardCol] === select.value);
    const cols = Object.keys(rows[0] || {{}});

    let tableHtml = "<table border='1' style='border-collapse: collapse; width: 100%; font-size: 13px;'>";
    tableHtml += "<thead><tr>" + cols.map(c => "<th style='padding:6px; background:#f2f2f2; text-align:left;'>" + c + "</th>").join("") + "</tr></thead>";
    tableHtml += "<tbody>";

    rows.forEach(function(row) {{
      tableHtml += "<tr>";
      cols.forEach(function(c) {{
        let value = row[c];
        if (value === null || value === undefined) value = "";
        const style = isNegativeNumber(value)
          ? "padding:6px; color:red; font-weight:600;"
          : "padding:6px;";
        tableHtml += "<td style='" + style + "'>" + value + "</td>";
      }});
      tableHtml += "</tr>";
    }});

    tableHtml += "</tbody></table>";
    tableDiv.innerHTML = tableHtml;
  }}

  select.addEventListener("change", renderTable);
  select.value = wards[0];
  renderTable();
}})();
</script>
"""

display(HTML(html))